# LLM Preview Workbench

Notebook isolé pour tester une réponse LLM au-dessus de la branche retrieval `phase8b`.

Ce notebook montre:
- la requête brute
- le stage 1 documentaire
- les chunks envoyés au LLM
- le prompt citations-first
- la réponse LLM si `OPENAI_API_KEY` est disponible

Ce notebook n'est **pas** une promotion produit.
Il sert à auditer un preview end-to-end en mode contrôlé.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from bofip_cleanroom.preview_runtime import Phase8bPreviewRuntime
from bofip_cleanroom.llm_preview import build_citation_prompt, generate_preview_answer, has_openai_api_key


In [2]:
QUERY = "Notre startup a le statut JEI et porte des travaux de recherche. Peut-elle recuperer sa creance de CIR tout de suite ?"
RUN_LLM = False
MODEL = "gpt-5.4-mini"


In [3]:
runtime = Phase8bPreviewRuntime.from_local_corpus(corpus="commentary", device="cpu")
retrieval = runtime.retrieve(QUERY, top_docs=5, chunks_per_doc=3, max_chunks=8)

{
    "query": retrieval.query,
    "lexical_query": retrieval.lexical_query,
    "acronym_expansions": retrieval.acronym_expansions,
    "source_confidences": retrieval.source_confidences,
}


{'query': 'Notre startup a le statut JEI et porte des travaux de recherche. Peut-elle recuperer sa creance de CIR tout de suite ?',
 'lexical_query': 'Notre startup a le statut JEI et porte des travaux de recherche. Peut-elle recuperer sa creance de CIR tout de suite ? jeunes entreprises innovantes credit impot recherche',
 'acronym_expansions': [{'acronym': 'JEI',
   'phrase': 'jeunes entreprises innovantes'},
  {'acronym': 'CIR', 'phrase': 'credit impot recherche'}],
 'source_confidences': {'base': 0.17113,
  'sections_leads': 0.235992,
  'sections_leads_stem': 0.242246,
  'dense': 0.178528,
  'chunk_dense': 0.152225}}

## Stage 1 documents


In [4]:
[
    {
        "rank": hit.rank,
        "score": round(hit.score, 6),
        "boi_reference": hit.boi_reference,
        "title": hit.title,
        "sources": hit.sources,
        "ranks": hit.ranks,
    }
    for hit in retrieval.stage1_hits
]


[{'rank': 1,
  'score': 0.182153,
  'boi_reference': 'BOI-BIC-CHAMP-80-20-20-20-20240703',
  'title': "BIC - Champ d'application et territorialité - Exonérations - Entreprises exerçant une activité particulière - Jeunes entreprises innovantes, jeunes entreprises universitaires et jeunes entreprises de croissance - Portée et calcul des allègements fiscaux",
  'sources': ['base',
   'chunk_dense',
   'dense',
   'sections_leads',
   'sections_leads_stem'],
  'ranks': {'base': 7,
   'sections_leads': 1,
   'sections_leads_stem': 1,
   'dense': 9,
   'chunk_dense': 2}},
 {'rank': 2,
  'score': 0.146889,
  'boi_reference': 'BOI-BIC-CHAMP-80-20-20-10-20250716',
  'title': 'BIC - Champ d’application et territorialité - Exonérations - Entreprises exerçant une activité particulière - Jeunes entreprises innovantes, jeunes entreprises universitaires et jeunes entreprises de croissance - Conditions d’éligibilité',
  'sources': ['base',
   'chunk_dense',
   'dense',
   'sections_leads',
   'section

## Stage 2 chunks envoyés au LLM


In [5]:
[
    {
        "citation_id": chunk.citation_id,
        "boi_reference": chunk.boi_reference,
        "title": chunk.title,
        "section_path": chunk.section_path,
        "chunk_kind": chunk.chunk_kind,
        "text": chunk.text[:1200],
    }
    for chunk in retrieval.stage2_chunks
]


[{'citation_id': 1,
  'boi_reference': 'BOI-BIC-CHAMP-80-20-20-20-20240703',
  'title': "BIC - Champ d'application et territorialité - Exonérations - Entreprises exerçant une activité particulière - Jeunes entreprises innovantes, jeunes entreprises universitaires et jeunes entreprises de croissance - Portée et calcul des allègements fiscaux",
  'section_path': "I. Nature des avantages > B. Articulation avec le bénéfice du crédit d'impôt pour dépenses de recherche (CIR) et du crédit d'impôt en faveur de la recherche collaborative (CICo)",
  'chunk_kind': 'paragraph_window',
  'text': "Le I de l' article 244 quater B du CGI et le I de l' article 244 quater B bis du CGI prévoient expressément que les entreprises exonérées d'impôt sur les bénéfices en application de l' article 44 sexies A du CGI peuvent également solliciter le bénéfice du CIR et du CICo. De plus, le 3° du II de l' article 199 ter B du CGI et le 3° du II de l' article 199 ter B bis du CGI prévoient la possibilité pour les J

## Prompt


In [6]:
prompt_text = build_citation_prompt(retrieval)
print(prompt_text)


Question utilisateur:
Notre startup a le statut JEI et porte des travaux de recherche. Peut-elle recuperer sa creance de CIR tout de suite ?

Extraits BOFiP fournis:
[1] BOI: BOI-BIC-CHAMP-80-20-20-20-20240703
Titre: BIC - Champ d'application et territorialité - Exonérations - Entreprises exerçant une activité particulière - Jeunes entreprises innovantes, jeunes entreprises universitaires et jeunes entreprises de croissance - Portée et calcul des allègements fiscaux
Date: 2024-07-03
Section: I. Nature des avantages > B. Articulation avec le bénéfice du crédit d'impôt pour dépenses de recherche (CIR) et du crédit d'impôt en faveur de la recherche collaborative (CICo)
Texte: Le I de l' article 244 quater B du CGI et le I de l' article 244 quater B bis du CGI prévoient expressément que les entreprises exonérées d'impôt sur les bénéfices en application de l' article 44 sexies A du CGI peuvent également solliciter le bénéfice du CIR et du CICo. De plus, le 3° du II de l' article 199 ter B d

## Réponse LLM


In [7]:
if RUN_LLM:
    preview = generate_preview_answer(retrieval, model=MODEL)
    print({"api_called": preview.api_called, "model": preview.model})
    print()
    print(preview.answer_text)
else:
    print({"api_key_present": has_openai_api_key(), "run_llm": RUN_LLM})
    print("Passe RUN_LLM = True si OPENAI_API_KEY est disponible.")


{'api_key_present': False, 'run_llm': False}
Passe RUN_LLM = True si OPENAI_API_KEY est disponible.
